In [2]:
import pandas as pd
import numpy as np
import torch
import json
import os
from sklearn.decomposition import PCA
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.datasets import fashion_mnist

# Fashion MNIST

In [4]:
# --- 1. LOAD DATASET FASHION-MNIST ---
print("Loading Fashion-MNIST dataset...")

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Ubah shape dari (N, 28, 28) menjadi (N, 784)
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

# --- 2. SETTING FOLDER OUTPUT ---
folder_output = 'data_preprocessed'
os.makedirs(folder_output, exist_ok=True)

# --- 3. MAPPING LABEL KE JSON ---
label_names = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot"
}

with open(os.path.join(folder_output, 'label_map.json'), 'w') as f:
    json.dump(label_names, f)

print("Label map saved.")

# --- 4. NORMALISASI & SIMPAN RAW DATA ---
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

np.savez_compressed(
    os.path.join(folder_output, 'fashion_raw_data.npz'),
    X_train=X_train_norm,
    X_test=X_test_norm,
    y_train=y_train,
    y_test=y_test
)

print(f"Raw data saved (Dimensi: {X_train_norm.shape[1]}).")

# --- 5. PCA & SIMPAN ---
n_components = [32, 64, 128]

for component in n_components:
    print(f"Processing PCA Dimensi {component}...")

    pca = PCA(n_components=component)

    X_train_pca = pca.fit_transform(X_train_norm)
    X_test_pca = pca.transform(X_test_norm)

    np.savez_compressed(
        os.path.join(folder_output, f'fashion_pca_{component}dim.npz'),
        X_train=X_train_pca,
        X_test=X_test_pca,
        y_train=y_train,
        y_test=y_test
    )

print("All PCA variants saved.")

# --- 6. SIMPAN DATA UNTUK VAE (.pt) ---
vae_data = {
    'train_data': torch.FloatTensor(X_train_norm),
    'train_labels': torch.LongTensor(y_train),
    'test_data': torch.FloatTensor(X_test_norm),
    'test_labels': torch.LongTensor(y_test)
}

torch.save(
    vae_data,
    os.path.join(folder_output, 'fashion_pytorch_data.pt')
)

print("PyTorch data for VAE saved.")

print("\nSEMUA DATA SIAP! Folder 'data_preprocessed' sudah terisi.")

Loading Fashion-MNIST dataset...
Label map saved.
Raw data saved (Dimensi: 784).
Processing PCA Dimensi 32...
Processing PCA Dimensi 64...
Processing PCA Dimensi 128...
All PCA variants saved.
PyTorch data for VAE saved.

SEMUA DATA SIAP! Folder 'data_preprocessed' sudah terisi.


# Feature Extraction using VAE

In [5]:
# --- 1. KONFIGURASI ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dims = [32, 64, 128]
epochs = 30
batch_size = 128

# Load Data dari file .pt
data_path = 'data_preprocessed/fashion_pytorch_data.pt'
data = torch.load(data_path)

train_loader = DataLoader(TensorDataset(data['train_data'], data['train_labels']), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(data['test_data'], data['test_labels']), batch_size=batch_size, shuffle=False)

# --- 2. ARSITEKTUR VAE (Input 784) ---
class FashionVAE(nn.Module):
    def __init__(self, latent_dim):
        super(FashionVAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 400),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logvar = nn.Linear(400, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 400),
            nn.ReLU(),
            nn.Linear(400, 784),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def loss_function(recon_x, x, mu, logvar):
    BCE = nn.functional.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

# --- 3. LOOP TRAINING & EKSTRAKSI ---
for dim in latent_dims:
    print(f"\n Training VAE Dimensi: {dim}")
    model = FashionVAE(dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # Training Loop
    model.train()
    for epoch in range(1, epochs + 1):
        train_loss = 0
        for batch_idx, (data_batch, _) in enumerate(train_loader):
            data_batch = data_batch.to(device)
            optimizer.zero_grad()
            recon_batch, mu, logvar = model(data_batch)
            loss = loss_function(recon_batch, data_batch, mu, logvar)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch} | Loss: {train_loss / len(train_loader.dataset):.4f}')

    # Ekstraksi Fitur Laten (z)
    model.eval()
    with torch.no_grad():
        # Train set
        mu_train, _ = model.encode(data['train_data'].to(device))
        # Test set
        mu_test, _ = model.encode(data['test_data'].to(device))
        
        # Simpan ke .npz agar formatnya sama dengan PCA & Raw
        np.savez_compressed(f'data_preprocessed/fashion_vae_{dim}dim.npz', 
                            X_train=mu_train.cpu().numpy(), 
                            X_test=mu_test.cpu().numpy(),
                            y_train=data['train_labels'].numpy(), 
                            y_test=data['test_labels'].numpy())
        
        # Simpan weights model (opsional)
        torch.save(model.state_dict(), f'data_preprocessed/vae_fashion_{dim}.pth')

print("\n SELESAI! Semua fitur VAE telah diekstrak.")


 Training VAE Dimensi: 32
Epoch 10 | Loss: 242.8839
Epoch 20 | Loss: 240.4329
Epoch 30 | Loss: 239.4966

 Training VAE Dimensi: 64
Epoch 10 | Loss: 243.0019
Epoch 20 | Loss: 240.5100
Epoch 30 | Loss: 239.6053

 Training VAE Dimensi: 128
Epoch 10 | Loss: 243.7293
Epoch 20 | Loss: 241.1309
Epoch 30 | Loss: 240.0097

 SELESAI! Semua fitur VAE telah diekstrak.
